# Phase 1 — Patching, Noising, Swap

The remaining three quarters of the causal-intervention battery (`vconf/exp2_patching.py`,
`exp3_noising.py`, `exp4_swap.py`), completing `notebooks_benzon/phase_1/steering.ipynb`'s
battery for the one target that actually showed a self-report signal —
**20 Questions x `impurity`** (`notebooks_benzon/phase_1/steering.ipynb` found `synonyms x
nuance_defined` and `list_elicitation x variety` both saturate to a single self-report class on
Qwen/reduced, with no variance to intervene on; the `paper` profile that could have checked
whether that's a reduced-model artifact isn't runnable here — `HF_TOKEN` has no accepted licence
for the gated `google/gemma-3-27b-it`).

**Same bug fixed in all three modules** as `exp1_steering.py`: `numeric_midpoints(cfg.prompt_kind)`
returns `None` for every categorical prompt, and `intervention_metrics` silently fell back to
`metrics.MIDPOINTS` — `CONFIDENCE`'s own 10-class array — for the `*_confidence`-labeled columns.
Fixed in `exp2_patching.run_patching`, `exp3_noising.run_noising`, and `exp4_swap.run_swap` to
derive midpoints from `cfg.sentiment` directly.

**Departure from `bands()`-based trial selection.** Every one of these three experiments'
original trial-selection helpers (`select_patching_trials`, `select_calibration_trials`,
`confidence_pools`) picks trials by whether their self-report lands in `sentiment.high_band`/
`low_band` — the two *named extreme* classes (`"Even"`/`"Lopsided"` for `impurity`). On this
200-turn run those bands hold only 8 and 1 trials respectively (class distribution: Lopsided 1,
Skewed 127, Balanced 64, Even 8) — nowhere near the ~50-200 per side these experiments were
designed around. Rather than force a patching/calibration/donor pool out of a single trial,
every high/low split below uses a **rank-based split** instead — the top/bottom 30% of trials by
raw `t.confidence` value — which uses the *actual* spread in the data (Skewed vs. Balanced
mostly) instead of just its two least-populated tails. `select_noising_trials`'s default
(`split=None`, a plain random sample of *all* trials) needs no such change and is used unmodified.


In [1]:
import json as jsonlib
import os
import pathlib
import sys
from dataclasses import replace

import numpy as np
import pandas as pd

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vconf").is_dir())
sys.path.insert(0, str(ROOT))

HERE = pathlib.Path.cwd()
with open(HERE / "config.json") as f:
    MODEL_CONFIG = jsonlib.load(f)
MODEL_NAME = MODEL_CONFIG["model"]
os.environ["VCONF_MODEL"] = MODEL_NAME
CACHE_DIR = HERE / "cache" / MODEL_NAME
OUT_DIR = HERE / "out" / MODEL_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

from vconf import activations
from vconf import config as cfgmod
from vconf import exp2_patching as S2
from vconf import exp3_noising as S3
from vconf import exp4_swap as S4
from vconf import notebook as nb
from vconf import pipeline
from vconf import twenty_questions as TQ
from vconf import results as R
from vconf.sentiment import IMPURITY

cfg = nb.run_config("gemma-categorical", sentiment=IMPURITY, name="phase1-pns-twentyq")
print(nb.describe(cfg))
device_map = "auto" if nb.profile() == "paper" else None
loaded = nb.open_model(cfg, device_map=device_map)


profile          : reduced
model            : Qwen/Qwen2.5-7B-Instruct (28 layers)
sentiment        : impurity  (ground truth: correctness)
prompt / dataset : categorical / triviaqa
layer sweep      : (0, 5, 11, 16, 22, 27)
trial counts     : {'steering': 24, 'patching': 24, 'noising': 32, 'swap': 24, 'attention': 24}
activation set   : 300   calibration set: 40
chat template    : True   attention impl: None
NOTE             : reduced profile — procedures, prompts, positions and
                   metrics follow the manual exactly, but the model and the
                   sample sizes are smaller than the paper's, so the numbers
                   here are not expected to match its reported values.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## Rebuild the `impurity` trials

Same 200-turn game data `steering.ipynb` played (`twenty_questions.py`, checkpointed to
disk keyed by model) — reloaded rather than replayed, and re-scored with a fresh
`pipeline.run_phase1` pass (greedy, deterministic, so this reproduces the same trials byte for
byte).

In [2]:
CHECKPOINT_PATH = CACHE_DIR / "twentyq-games.json"
assert CHECKPOINT_PATH.exists(), f"no checkpointed games at {CHECKPOINT_PATH} -- run steering.ipynb first"

all_records = jsonlib.loads(CHECKPOINT_PATH.read_text())
print(f"loaded {len(all_records)} turns from {CHECKPOINT_PATH}")

impurity_trials = []
for record in all_records:
    state = TQ.transcript_summary(tuple(record["keywords"]), record["qa_history_before"])
    qid = f"{record['category']}_game{record['game']}_turn{record['turn']}"
    impurity_trials.append(pipeline.Trial(qid=qid, question=state, answer=record["question"]))

rendered = pipeline.run_phase1(loaded, impurity_trials, cfg)
trials = impurity_trials
print(f"{len(trials)} trials scored, self-report range "
      f"{min(t.confidence for t in trials):.2f}..{max(t.confidence for t in trials):.2f}")


loaded 200 turns from /scratch/qi/project/results/trials/phase1-steering-twentyq-games-qwen.json


200 trials scored, self-report range 0.12..0.88


In [3]:
def split_high_low(trials, frac=0.3):
    """Top/bottom ``frac`` of trials by raw self-report value -- see the departure-from-`bands()`
    note above."""
    order = np.argsort([t.confidence for t in trials])
    k = max(1, int(len(trials) * frac))
    return order[-k:][::-1].copy(), order[:k].copy()  # high (descending), low (ascending)


high_idx, low_idx = split_high_low(trials, frac=0.3)
print(f"high pool: {len(high_idx)} trials, confidence {trials[high_idx[-1]].confidence:.2f}"
      f"..{trials[high_idx[0]].confidence:.2f}")
print(f"low pool: {len(low_idx)} trials, confidence {trials[low_idx[0]].confidence:.2f}"
      f"..{trials[low_idx[-1]].confidence:.2f}")


high pool: 60 trials, confidence 0.62..0.88
low pool: 60 trials, confidence 0.12..0.38


## Patching (§5, `exp2_patching.py`)

Corrupt the model's own generated question (the "answer" span of the Phase-1 impurity prompt) by
mean-ablating its input embeddings, then restore one position's clean residual stream at a time
and see how much of the clean impurity report comes back. Test trials: the high pool (impurity
should have somewhere to drop *from*); calibration for the mean-ablation embedding: the combined
high+low pool.

In [4]:
PATCH_POSITIONS = ("PANL", "PANL+1", "CC")

test_idx_patch = high_idx[:24]
calibration_idx_patch = np.concatenate([high_idx, low_idx])
test_rendered = [rendered[i] for i in test_idx_patch]
test_trials = [trials[i] for i in test_idx_patch]
calibration_rendered = [rendered[i] for i in calibration_idx_patch]

clean_store = activations.collect_activations(
    loaded, test_rendered, cfg.layers, PATCH_POSITIONS,
    trial_ids=[t.qid for t in test_trials], batch_size=cfg.batch_size,
)
mean_embeddings = S2.compute_mean_answer_embeddings(loaded, calibration_rendered, batch_size=cfg.batch_size)

patch_frame, patch_baselines = S2.run_patching(
    loaded, test_rendered, test_trials, clean_store, mean_embeddings, cfg=cfg,
    positions=PATCH_POSITIONS,
)
print({k: v for k, v in patch_baselines.items() if k != "corrupt_frame"})
recovery = S2.recovery_table(patch_frame, patch_baselines)
display(recovery)


{'clean_logit_diff': 5.6163194444444455, 'corrupt_logit_diff': 1.690972222222222, 'clean_confidence': 0.7066666666666667, 'corrupt_confidence': 0.555, 'corrupt_token_change_rate': 0.7083333333333334}


,layer,position,logit_diff,confidence,token_change_rate,logit_diff_recovery,confidence_recovery,token_change_recovery
0,0,CC,1.709201,0.555000,0.708333,0.464396,0.000000,0.000000
1,5,CC,1.659722,0.555000,0.708333,-0.796108,0.000000,0.000000
2,11,CC,1.578125,0.565833,0.666667,-2.874834,7.142857,5.882353
3,16,CC,1.791667,0.576667,0.625000,2.565237,14.285714,11.764706
4,22,CC,4.734375,0.666667,0.250000,77.532065,73.626374,64.705882
5,27,CC,5.616319,0.706667,0.000000,100.000000,100.000000,100.000000
6,0,PANL,1.543403,0.565000,0.541667,-3.759398,6.593407,23.529412
7,5,PANL,0.854167,0.513333,0.750000,-21.318001,-27.472527,-5.882353
8,11,PANL,1.352431,0.534167,0.750000,-8.624502,-13.736264,-5.882353
9,16,PANL,1.755208,0.523333,0.708333,1.636444,-20.879121,0.000000


In [5]:
peak_patch = recovery.loc[recovery.groupby("position")["confidence_recovery"].idxmax()]
display(peak_patch[["position", "layer", "confidence_recovery", "logit_diff_recovery", "token_change_recovery"]])


,position,layer,confidence_recovery,logit_diff_recovery,token_change_recovery
5,CC,27,100.000000,100.000000,100.000000
6,PANL,0,6.593407,-3.759398,23.529412
14,PANL+1,11,7.142857,-4.422822,5.882353


## Noising (§6, `exp3_noising.py`)

Replace one position's residual stream with the *mean* activation of the high+low calibration
pool (disruption, not a push toward "neutral"), one (layer, position) at a time, tested on a
plain random sample of all 200 trials — `select_noising_trials`'s default path needs no
rank-split override since it never touches `sentiment.high_band`/`low_band` at all.

In [6]:
NOISE_POSITIONS = ("PANL", "PANL+1", "CC")

means = S3.mean_activations(loaded, calibration_rendered, cfg=cfg, positions=NOISE_POSITIONS)

noise_test_idx = S3.select_noising_trials(trials, n=40, seed=0)
noise_rendered = [rendered[i] for i in noise_test_idx]
noise_trials = [trials[i] for i in noise_test_idx]

noise_frame = S3.run_noising(loaded, noise_rendered, noise_trials, means, cfg=cfg, positions=NOISE_POSITIONS)
noise_summary = R.summarize(noise_frame, by=("position", "layer"))
display(noise_summary[["position", "layer", "n", "logit_diff_change_mean", "token_changed_mean"]])


,position,layer,n,logit_diff_change_mean,token_changed_mean
0,CC,0,40,0.007292,0.000
1,CC,5,40,0.014583,0.000
2,CC,11,40,0.020833,0.025
3,CC,16,40,0.031250,0.025
4,CC,22,40,-3.929167,0.575
5,CC,27,40,-4.412500,0.575
6,PANL,0,40,-0.028125,0.075
7,PANL,5,40,-0.136458,0.100
8,PANL,11,40,-0.068750,0.100
9,PANL,16,40,-0.037500,0.075


In [7]:
peak_noise = noise_summary.loc[noise_summary.groupby("position")["logit_diff_change_mean"].apply(lambda s: s.abs().idxmax())]
display(peak_noise[["position", "layer", "logit_diff_change_mean", "token_changed_mean"]])


,position,layer,logit_diff_change_mean,token_changed_mean
5,CC,27,-4.412500,0.575
7,PANL,5,-0.136458,0.100
13,PANL+1,5,-0.078125,0.075


## Swap (§7, `exp4_swap.py`)

Transplant one trial's PANL (etc.) activation into a *different* trial's forward pass and see
whether the borrowed impurity level comes along. `build_swap_design` is reimplemented inline
below with the same body as `exp4_swap.build_swap_design`, just parameterized on the rank-split
`high_idx`/`low_idx` computed above instead of calling `confidence_pools` (which reads
`sentiment.high_band`/`low_band` internally, the same bands section above found too thin).
`sample_recipients` already samples with replacement once a pool is shorter than the requested
`n`, so the small pools here still produce a full design, just with repeated recipients.

In [8]:
def build_swap_design_ranked(trials, rendered, high_idx, low_idx, n, seed=0,
                              n_bins=cfgmod.DONOR_QUANTILE_BINS):
    q_len, a_len = S4.prompt_lengths(rendered)
    recipients = {
        "H": S4.sample_recipients(high_idx, n, seed=seed),
        "L": S4.sample_recipients(low_idx, n, seed=seed + 1),
    }
    design, stats = {}, {}
    for condition in S4.CONDITIONS:
        recipient_band, donor_band = condition.split("->")
        donor_pool = high_idx if donor_band == "H" else low_idx
        donors, quality = S4.match_donors(
            recipients[recipient_band], donor_pool, q_len, a_len, n_bins=n_bins,
            seed=seed + hash(condition) % 1000,
        )
        design[condition] = (recipients[recipient_band], donors)
        stats[condition] = quality
    return design, stats


SWAP_POSITIONS = ("PANL", "PANL+1", "CC")
swap_design, swap_stats = build_swap_design_ranked(trials, rendered, high_idx, low_idx, n=24, n_bins=3)
display(pd.DataFrame(swap_stats).T)

donor_store = activations.collect_activations(
    loaded, rendered, cfg.layers, SWAP_POSITIONS,
    trial_ids=[t.qid for t in trials], batch_size=cfg.batch_size,
)
swap_frame = S4.run_swap(loaded, rendered, trials, swap_design, donor_store, cfg=cfg, positions=SWAP_POSITIONS)
swap_summary = R.summarize(swap_frame, by=("position", "condition", "layer"))


,question_bin_match,answer_bin_match,mean_abs_question_delta,mean_abs_answer_delta
H->H,1.0,1.0,34.750000,3.333333
L->L,1.0,1.0,30.958333,4.125000
H->L,1.0,1.0,29.458333,3.166667
L->H,1.0,1.0,37.958333,4.791667


In [9]:
cross_minus_same = S4.cross_minus_same(swap_summary, metric="confidence_change")
display(cross_minus_same)


,layer,position,L->H minus L->L,H->L minus H->H
0,0,CC,0.0000,0.000000
1,5,CC,0.0000,0.000000
2,11,CC,0.0000,0.000000
3,16,CC,0.0000,0.000000
4,22,CC,0.2625,-0.261667
5,27,CC,0.2725,-0.294167
6,0,PANL,0.0000,0.010833
7,5,PANL,0.0000,-0.010833
8,11,PANL,0.0000,-0.020833
9,16,PANL,0.0000,-0.010000


In [10]:
peak_swap = swap_summary.loc[
    swap_summary.groupby(["position", "condition"])["confidence_change_mean"].apply(lambda s: s.abs().idxmax())
]
display(peak_swap[["position", "condition", "layer", "n", "confidence_change_mean", "logit_diff_change_mean"]])


,position,condition,layer,n,confidence_change_mean,logit_diff_change_mean
4,CC,H->H,22,24,-0.021667,-0.777778
11,CC,H->L,27,24,-0.305000,-9.510417
17,CC,L->H,27,24,0.272500,-8.321181
18,CC,L->L,0,24,0.000000,-0.008681
27,PANL,H->H,16,24,-0.030833,-0.192708
33,PANL,H->L,16,24,-0.040833,-0.322917
36,PANL,L->H,0,24,0.000000,0.024306
42,PANL,L->L,0,24,0.000000,0.019097
49,PANL+1,H->H,5,24,-0.010833,-0.052083
54,PANL+1,H->L,0,24,-0.010833,-0.027778


## Combined picture, all four interventions on `twenty_questions x impurity`

`steering.ipynb`'s steering result: PANL peaks at an earlier layer than CC (L2.5 vs
L13.5, |delta| 0.022 vs 0.077) — the qualitative "PANL before CC" signature. The cells above ask
the same question three more ways: does restoring PANL recover more of the corrupted signal than
CC does at an earlier layer (patching); does mean-ablating PANL disrupt the signal comparably to
CC (noising); does a donor's PANL activation carry the borrowed impurity level into an unrelated
recipient (swap, the cross-confidence conditions `L->H`/`H->L` vs. their same-confidence
controls `L->L`/`H->H`)? Read the four peak tables above together rather than any one in
isolation — on a 200-trial, single-model run each individual number carries real sampling
noise.